#Autonomous Driving Environment

## Environment

In [ ]:
#environment.py

import random
import sys

MAX_VIEW = 10 # DO NOT CHANGE THIS
INIT_LOCATION_WEIGHT = 1/5 # DO NOT CHANGE THIS

class Environment:
    def __init__(self, height=10, width=5, occupancy=0.1, init=0.2, seed=5):
        self.height = height
        self.width = width
        self.occupancy = occupancy
        self.init = init
        self.cars = {}
        self.road = [[0 for _ in range(width)] for _ in range(height)]
        self.crash = None
        self.done = None

        self.actions = ['F', 'L', 'R', 'W']
        self.score = 0
        self.__generate_environment(seed)

    def __generate_environment(self, seed):
        """
        Description
        -----------
        This function is used to populate self.road and self.cars
        self.cars is a dict that holds the location of all cars on the road including the autonomous agent's (idx 0)
        self.road is a 2D list (grid) representing the discretized road environment with the following markers:
            0: empty cell
            1: obstacle car
            2: autonomous agent (AA)
           -1: crash

        The convention for positions, like an array, is that (r,c) = (0,0) is the lower left corner, r increases
        vertically and c increases horizontally.

        """
        random.seed(seed)

        # Number of obstacles should not exceed OCCUPANCY cells in the environment
        n_obstacles = int(self.height * self.width * self.occupancy)

        coordinate_list = [(r, c) for r in range(self.height) for c in range(self.width)]
        obstacle_list = random.sample(coordinate_list, n_obstacles)

        # Mark the AA
        r = int(self.height * self.init)
        c = int(self.width * self.init)
        self.road[r][c] = 2
        self.cars[0] = (r, c) #AA location is stored in the 0th index of cars dict

        # Mark obstacle cars
        idx = 1
        for r, c in obstacle_list:
            if (r, c) != self.cars[0]:
                self.road[r][c] = 1
                self.cars[idx] = (r, c)
                idx += 1

    def get_next_cell(self, idx, action):
        """
        Description
        -----------
        This function finds a car's next location given a car's index and action it may perform.
        If the action performed leads a car outside the road, its next location is updated to None

        """
        loc = self.cars[idx]

        if action == 'F':
            if loc[0]+1 < self.height:
                return (loc[0]+1, loc[1])
            elif loc[0]+1 == self.height: # moving forward leads out of the road
                return None
        elif action == 'L':
            return (loc[0], loc[1]-1)
        elif action == 'R':
            return (loc[0], loc[1]+1)
        elif action == 'W':
            return loc

    def get_possible_next_cells(self, idx):
        """
        Description
        -----------
        This function finds all possible next locations given a car's index (except AA's).

        """
        next_cells = []
        for action in self.get_legal_actions(idx):
            next_loc = self.get_next_cell(idx, action)
            next_cells.append(next_loc)

        return next_cells

    def get_legal_actions(self, idx):
        """
        Description
        -----------
        This function finds all possible legal actions given a car's index by considering its current location.
        Actions leading to locations where a car (except AA) is already present is considered illegal for other cars.
        Wait action is considered legal by default for other cars.

        """
        loc = self.cars[idx]
        legal_actions = []
        # Check all three directions a car can move to
        if loc[0]+1 < self.height and self.road[loc[0]+1][loc[1]] != 1: # Forward action
            legal_actions.append('F')
        elif loc[0]+1 == self.height: # moving forward leads out of the road
            legal_actions.append('F')

        if loc[1]-1 >= 0 and self.road[loc[0]][loc[1]-1] != 1: # Left action
            legal_actions.append('L')

        if loc[1]+1 < self.width and self.road[loc[0]][loc[1]+1] != 1: # Right action
            legal_actions.append('R')

        legal_actions.append('W') # Wait action

        return legal_actions

    def __update_environment(self):
        """
        Description
        -----------
        This function updates the position of all cars (excluding 'A''s) in the environment everytime the step function is called.
        A random action is chosen among the available legal actions
        Cars that move forwad beyond the scope of the road are no longer considered part of the environment and their location updates to None.

        """

        for idx, loc in self.cars.items():
            if idx == 0 or loc == None:
                continue
            legal_actions = self.get_legal_actions(idx)
            action = random.choice(legal_actions)
            self.perform_car_action(idx, action)

    def perform_car_action(self, idx, action):
        """
        Description
        -----------
        This function updates the position of a car in self.cars (except AA) as a result of the action performed in the environment.

        """
        loc = self.cars[idx]
        next_cell = self.get_next_cell(idx, action)
        if next_cell == None: # car has crossed the road
            self.road[loc[0]][loc[1]] = 0 # Erase current marker
        else:
            self.road[loc[0]][loc[1]] = 0 # Erase current marker
            self.road[next_cell[0]][next_cell[1]] = 1 # Add new marker

        self.cars[idx] = next_cell # update location of car in self.cars

    def perform_AA_action(self, action):
        """
        Description
        -----------
        This function updates the position of the AA's as a result of the action performed in the environment everytime the step function is called.

        """
        loc = self.cars[0]
        if action == 'F':
            if loc[0]+1 < self.height: # Next cell is within the scope of the road
                if self.road[loc[0]+1][loc[1]] == 1: # 'A' is crashing into a car
                    self.crash = True
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]+1][loc[1]] = -1 # Add marker for crash
                    self.cars[0] = (-1, -1)
                else:
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]+1][loc[1]] = 2 # Add new marker
                    self.cars[0] = (loc[0]+1, loc[1])
                    self.score += 1
            else: # 'A' has successfully navigated till the end of the road
                self.done = True
                self.cars[0] = None
        elif action == 'L':
            if loc[1]-1 >= 0: # Next cell is within the scope of the road
                if self.road[loc[0]][loc[1]-1] == 1: # A is crashing into a car
                    self.crash = True
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]-1] = -1 # Add marker for crash
                    self.cars[0] == (-1, -1)
                else:
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]-1] = 2 # Add new marker
                    self.cars[0] = (loc[0], loc[1]-1)
            else: # 'A' is moving out of the scope of the road (i.e. driving beyond the left shoulder of the road)
                self.crash = True
                if self.road[loc[0]][loc[1]] != 1:
                    self.road[loc[0]][loc[1]] = 0 # Erase current marker
                self.cars[0] == (-1, -1)
        elif action == 'R':
            if loc[1]+1 < self.width: # Next cell is within the scope of the road
                if self.road[loc[0]][loc[1]+1] == 1: # A is crashing into a car
                    self.crash = True
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]+1] = -1 # Add marker for crash
                    self.cars[0] == (-1, -1)
                else:
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]+1] = 2 # Add new marker
                    self.cars[0] = (loc[0], loc[1]+1)
            else: # 'A' is moving out of the scope of the road (i.e. driving beyond the right shoulder of the road)
                self.crash = True
                if self.road[loc[0]][loc[1]] != 1:
                    self.road[loc[0]][loc[1]] = 0 # Erase current marker
                self.cars[0] == (-1, -1)
        elif action == 'W':
            if self.road[loc[0]][loc[1]] == 1: # A is crashing into a car
                self.crash = True
                self.road[loc[0]][loc[1]] = 0 # Erase current marker
                self.road[loc[0]][loc[1]] = -1 # Add marker for crash
                self.cars[0] == (-1, -1)
        elif action == 'S':
            self.done = False
        else:
            raise ValueError(f"Incorrect action specified: {action}")

    def get_percept(self):
        """
        Description
        -------
        This function returns the percept - a (3 x 3) grid sized partial view of the road environment,
        where AA's current location is at the center of the grid. It has the following markers:
            0: empty cell
            1: obstacle car
            2: autonomous agent (AA)
            -1: edge of the road (left, right and bottom edge of the road)
            10: end of the road (top edge of the road)

        """
        partial_state = [[0 for _ in range(3)] for _ in range(3)]

        AA_loc = self.cars[0]
        for x, r in enumerate(range(AA_loc[0]+1, AA_loc[0]-2, -1)):
            for y, c in enumerate(range(AA_loc[1]-1, AA_loc[1]+2, 1)):
                if r < 0:
                    partial_state[x][y] = -1
                elif r == self.height:
                    partial_state[x][y] = 10
                elif c < 0 or c == self.width:
                    partial_state[x][y] = -1
                else:
                    partial_state[x][y] = self.road[r][c]

        return partial_state

    def step(self, action):
        """
        Description
        -----------
        Execute the action taken by the AA.
        Agents's actions are deterministic. However, environment's stochasticity lies in the uncerternity of location of other cars.

        """
        self.__update_environment()
        self.perform_AA_action(action)
        if self.crash == True:
            print("There is a crash on the road!")
            self.display_environment()
            sys.exit(0)
        elif self.done == True:
            print("AA has successfully navigated the road!")
            sys.exit(0)
        elif self.done == False:
            print("Stopping simulation!")
            sys.exit(0)
        # self.display_environment()
        # self.display_percept()

    def display_environment(self):
        """
        Description
        -----------
        Display the environment with static view of the cars including the autonomous agent
        Example: Change in autonomous driving agent's location.

        You may use this for debugging only. The AA may not have access to the actual state as displayed in this function.

        """

        display = ""
        for r in range(self.height-1, -1, -1):
            # Display borders
            display += chr(43)
            for c in range(self.width):
                display += chr(45) + chr(45) + chr(45) + chr(43)
            display += "\n"

            # Display road
            display += chr(124)
            for c in range(self.width):
                if self.road[r][c] == 1:
                    # ASCII 94 = ^
                    display += chr(32) + chr(94) + chr(32) + chr(124)
                elif self.road[r][c] == 2:
                    # ASCII 65 = A
                    display += chr(32) + chr(65) + chr(32) + chr(124)
                elif self.road[r][c] == -1:
                    # ASCII 42 = *
                    display += chr(32) + chr(42) + chr(32) + chr(124)
                else:
                    # ASCII 32 = \space
                    display += chr(32) + chr(32) + chr(32) + chr(124)
            display += "\n"

        # Display border
        display += chr(43)
        for c in range(self.width):
            display += chr(45) + chr(45) + chr(45) + chr(43)
        display += "\n"

        print(display)
        return

    def display_percept(self):
        """
        Description
        -----------
        Display the (3 x 3) grid sized percept of the AA based on its current location.

        """
        percept = self.get_percept()

        height, width = len(percept), len(percept[0])
        display = ""
        for r in range(height):
            # Display borders
            display += chr(43)
            for c in range(width):
                display += chr(45) + chr(45) + chr(45) + chr(43)
            display += "\n"

            # Display road
            display += chr(124)
            for c in range(width):
                if percept[r][c] == 1:
                    # ASCII 94 = ^
                    display += chr(32) + chr(94) + chr(32) + chr(124)
                elif percept[r][c] == 2:
                    # ASCII 65 = A
                    display += chr(32) + chr(65) + chr(32) + chr(124)
                elif percept[r][c] == -1:
                    # ASCII 120 = x
                    display += chr(32) + chr(120) + chr(32) + chr(124)
                elif percept[r][c] == 10:
                    # ASCII 71 = G
                    display += chr(32) + chr(71) + chr(32) + chr(124)
                else:
                    # ASCII 32 = \space
                    display += chr(32) + chr(32) + chr(32) + chr(124)
            display += "\n"

        # Display border
        display += chr(43)
        for c in range(width):
            display += chr(45) + chr(45) + chr(45) + chr(43)
        display += "\n"

        print(display)
        return

## State

In [ ]:
# state.py

import copy

"""
A state must define methods such as get_percept, generate_successor, get_legal_actions, etc. as required
depending on the type of agent for which an interface is being built.
States are used by the AutonomousDriving object to capture the actual state of the environment/road and
it can be used by agents to reason about driving.
Every agent type has a different state information available to it.
This is a super class for all types of state information.

"""

class ManualState():
    """
    The full state or percept is displayed to a user engaging in manual control.
    You may modify this at your will but it is unnecessary.

    """

    def __init__(self, env):
        self.__env = env

    def display_state(self):
        # self.__env.display_environment() # Comment this if you do not wish to display the road at every step
        self.__env.display_percept() # Comment this if you do not wish to display the percept at every step
        pass

class RandomState():
    """
    The full state or percept is displayed when random agent control is chosen.
    You may modify this at your will but it is unnecessary.

    """

    def __init__(self, env):
        self.__env = env

    def display_state(self):
        # self.__env.display_environment() # Comment this if you do not wish to display the road at every step
        self.__env.display_percept() # Comment this if you do not wish to display the percept at every step
        print(self.__env.get_percept())
        pass

class ExpectimaxState():
    """
    An expectimax agent chooses an action at each choice point by evaluating the expected return for choosing from its available actions.
    Helper functions are defined to access the attributes of the environment.
    We strongly recommend that you use the following functions to write your agent function and not access the environment variable directly.

    """

    def __init__(self, env):
        self.__env = env

    def get_legal_actions(self, car_index):
        """
        Description
        -------
        Returns all the legal actions for the given car.

        """
        if car_index == 0: # AA
            return self.__env.actions # All actions are legal for the AA
        elif car_index in self.__env.cars.keys():
            car_loc = self.__env.cars[car_index]
            if car_loc == None:
                return []
            else:
                return self.__env.get_legal_actions(car_index)
        else:
            raise Exception(f"Invalid car index specified: {car_index}")

    def generate_successor(self, car_index, action):
        """
        Description
        -------
        Returns the successor state after the specified car takes the action.

        """
        # Check that successors exist
        if self.is_done() or self.is_crash():
            raise Exception('Cannot generate a successor of a terminal state.')
        elif self.__env.cars[car_index] == None:
            raise Exception('Cannot generate a successor of a state with car index that is no longer on the road.')

        # Copy current state
        state = copy.deepcopy(ExpectimaxState(self.__env))

        # Let car's logic deal with its action's effects on the road
        if car_index == 0:  # AA is moving
            state.apply_AA_action(action)
        else:               # Other car is moving
            state.apply_action(car_index, action)

        return state

    def apply_AA_action(self, action):
        """
        Description
        -------
        Simulates the action that is performed by the AA.

        """
        self.__env.perform_AA_action(action)

    def apply_action(self, car_index, action):
        """
        Description
        -------
        Simulates the action that is performed by the given car.

        """
        self.__env.perform_car_action(car_index, action)

    def get_num_cars(self):
        """
        Description
        -------
        Returns the number of cars initialized on the road.
        Note that this also includes count of cars that may have already crossed the road.

        """
        return len(self.__env.cars.items())

    def get_car_position(self, car_index):
        """
        Description
        -------
        Returns the location of the given car in the form of a tuple (row, col)

        """
        return self.__env.cars[car_index]

    def get_min_distance_to_goal(self, AA_loc):
        """
        Description
        -------
        Returns the shortest distance to cross the road by the AA.

        """
        return self.__env.height - AA_loc[0]

    def get_score(self):
        """
        Description
        -------
        Returns the score of a given state.

        """
        return self.__env.score

    def is_car_on_road(self, car_index):
        """
        Description
        -------
        Returns a boolean indicating if the given car is still on the road.

        """
        if car_index in self.__env.cars.keys():
            if self.__env.cars[car_index] == None:
                return False
            else:
                return True
        else:
            raise Exception(f"Invalid car index specified: {car_index}")

    def is_done(self):
        """
        Description
        -------
        Returns a boolean indicating if the AA has successfully crossed the road.

        """
        return self.__env.done

    def is_crash(self):
        """
        Description
        -------
        Returns a boolean indicating if there is a crash on the road.

        """
        return self.__env.crash


## Agent

In [ ]:
import random
import numpy as np

class Agent:
    """
    An agent must define a get_action method, but may also define
    other methods which will be called if they exist.
    This is a super class for any agent type.

    """

    def __init__(self):
        """
        Description
        -----------
        The list of available actions for all cars are:
        Forward - 'F'
        Left - 'L'
        Right - 'R'
        Wait - 'W'

        """
        self.available_actions = ['F', 'L', 'R', 'W']

    def get_action(self, state):
        """
        The Agent will receive a State of the environment (based on the agent type) and
        must return an action from the available actions {Forward - 'F', Left - 'L', Right - 'R' and Wait - 'W'}.
        """
        pass

#### helper functions

In [ ]:
def normalize(value, min_value, max_value):
    """
    Description
    ----------
    Normalizes a given value between 0-1

    """
    return (value - min_value) / (max_value - min_value)

def manhattan_distance(loc1, loc2):
    """
    Description
    ----------
    Returns the Manhattan distance between points loc1 and loc2

    """
    return abs( loc1[0] - loc2[0] ) + abs( loc1[1] - loc2[1] )

### Manual Agent

In [ ]:
class ManualAgent(Agent):
    """
    A manual agent is used to control the Autonomous Agent ('A') manually by the user.

    """

    def get_action(self, percept):
        "*** YOUR CODE HERE ***"
        print("Enter Action (Forward - 'F', Left - 'L', Right - 'R', Wait - 'W', Stop - 'S'):\n")
        action = input()
        return action

### Random Agent

In [ ]:
class RandomAgent(Agent):
    """
    A random agent chooses an action randomly at each choice point from the list of available actions.

    """

    def get_action(self, percept):
        "*** YOUR CODE HERE ***"
        action = random.choice(self.available_actions) #should choose an action among the legal actions available
        print(f"Random action chosen: {action}")
        input("Press enter to step through.")
        return action


### Expectimax Agent

In [ ]:
from re import A
class ExpectimaxAgent(Agent):
    """
    An expectimax agent chooses an action at each choice point based on the expectimax algorithm.
    The choice is dependent on the self.evaluationFunction.

    All other cars should be modeled as choosing uniformly at random from their legal actions.

    """
    def __init__(self, depth=3):
        self.index = 0 # 'A' is always agent index 0
        self.depth = int(depth)
        super().__init__()

    def evaluation_function(self, road_state):
        """
        Description
        ----------
        This function returns a score (float) given a state of the road.

        """
        "*** YOUR CODE HERE ***"
        if road_state.is_crash():
          return -999
        if road_state.is_done():
          return 999
        aa = road_state.get_car_position(0)
        val = -road_state.get_min_distance_to_goal(aa) * 10
        for i in range(1, road_state.get_num_cars()):
          if road_state.is_car_on_road(i):
            p = road_state.get_car_position(i)
            d = manhattan_distance(aa, p)
            if d <= 2:
              val -= (3 - d) * 20
        return val

    def get_action(self, road_state):
        """
        Description
        ----------
        This function returns the expectimax action using self.depth and self.evaluationFunction.
        All other cars should be modeled as choosing uniformly at random from their
        legal moves.

        """
        "*** YOUR CODE HERE ***"
        n = road_state.get_num_cars()

        def value(state, depth, agent):
          if depth == self.depth or state.is_crash() or state.is_done():
            return self.evaluation_function(state)
          nxt = (agent + 1) % n
          nd = depth + 1 if nxt == 0 else depth
          if agent == 0:
            return max(value(state.generate_successor(0,a), nd, nxt) for a in state.get_legal_actions(0))
          acts = state.get_legal_actions(agent)
          if not acts:
            return value(state, nd, nxt)
          return sum(value(state.generate_successor(agent, a), nd, nxt) for a in acts) / len(acts)

          best_v = float('-inf')
          best_a = 'W'
          nxt = 1 % n
          nd = 0 if n > 1 else 1
          for a in road_state.get_legal_actions(0):
            v = value(road_state.generate_successor(0, a), nd, nxt)
            if v > best_v:
              best_v = v
              best_a = a
          return best_a

## Problem

In [ ]:
# from environment import Environment
# from agent import *
# from state import *

import copy
import random


class AutonomousDriving:
    """
    This class creates a problem instance by initializing a specific agent type and simulates the autonomous driving environment.

    """
    def __init__(self, agent_type, height, width, occupancy, init, seed, custom_weights=False, weights=None):
        self.agent = None
        self.env = Environment(height, width, occupancy, init, seed)

        if agent_type == 'manual':
            self.agent = ManualAgent()
        elif agent_type == 'random':
            self.agent = RandomAgent()
        elif agent_type == 'expectimax':
            self.agent = ExpectimaxAgent()
        else:
            raise ValueError(f"Incorrect agent specified: {agent}")

    def run(self, agent_type, *args):
        """
        Description
        -----------
        This function calls the simulation function based on the agent initialized.
        and passes it to the step function.
        Thus, simulating the autonomous driving.

        """

        if agent_type == 'manual':
            self.run_manual_control()
        elif agent_type == 'random':
            self.run_random_control()
        elif agent_type == 'expectimax':
            self.run_expectimax_control()

    def run_manual_control(self):
        self.env.display_environment()
        while True:
            state_obj = ManualState(copy.deepcopy(self.env))
            state_obj.display_state()
            action = self.agent.get_action(None)
            self.env.step(action)

    def run_random_control(self):
        self.env.display_environment()
        while True:
            state_obj = RandomState(copy.deepcopy(self.env))
            state_obj.display_state()
            action = self.agent.get_action(None)
            self.env.step(action)

    def run_expectimax_control(self):
        self.env.display_environment()
        while True:
            state_obj = ExpectimaxState(copy.deepcopy(self.env))
            action = self.agent.get_action(state_obj)
            print(action)
            self.env.step(action)


## Playground

#### Feel free to play around with different environment configurations by changing the following parameters.
```
height - Road height (int), default: 10
width - Road width (int), default: 4
occupancy - Occupancy of the road with other cars. Enter a value between 0 and 0.3, default: 0.1
init - Initial location of 'A' w.r.t. the height of the road. \nENter a value between 0 and 0.9, default: 0.2
seed - To initialize cars on the road (int), default: 3
```

In [ ]:
height = 5
width = 4
occupancy = 0.1
init = 0.2
seed = 3

### Playground - Manual Agent

In [ ]:
fagent = 'manual'

driver = AutonomousDriving(
        agent_type=agent,
        height=height,
        width=width,
        occupancy=occupancy,
        init=init,
        seed=seed
    )

driver.run(agent)

+---+---+---+---+
|   |   | ^ |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
| A |   |   | ^ |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+

+---+---+---+
| x |   |   |
+---+---+---+
| x | A |   |
+---+---+---+
| x |   |   |
+---+---+---+

[[-1, 0, 0], [-1, 2, 0], [-1, 0, 0]]
Random action chosen: L


KeyboardInterrupt: Interrupted by user

### Playground - Random Agent

In [ ]:
agent = 'random'

driver = AutonomousDriving(
        agent_type=agent,
        height=height,
        width=width,
        occupancy=occupancy,
        init=init,
        seed=seed
    )

driver.run(agent)

+---+---+---+---+
|   |   | ^ |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
| A |   |   | ^ |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+

+---+---+---+
| x |   |   |
+---+---+---+
| x | A |   |
+---+---+---+
| x |   |   |
+---+---+---+

[[-1, 0, 0], [-1, 2, 0], [-1, 0, 0]]
Random action chosen: L


KeyboardInterrupt: Interrupted by user

### Playground - Expectimax Agent

In [ ]:
agent = 'expectimax'

# Checks for input parameters
assert agent == 'manual' or 'random' or 'expectimax', "agent should be one of the defined types (manual, random, expectimax)"
assert int(height) <= 5, "height of the road is too large for expectimax agent"
assert int(width) <= 5, "width of the road is too large for expectimax agent"
assert float(occupancy) <= 0.1, "occupancy of the road is too large for expectimax agent"
assert float(init) >= 0 and float(init) <= 0.9, "init is out of bounds"
assert type(int(seed)) is int, "seed should be an integer"

driver = AutonomousDriving(
        agent_type=agent,
        height=height,
        width=width,
        occupancy=occupancy,
        init=init,
        seed=seed
    )

driver.run(agent)

+---+---+---+---+
|   |   | ^ |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
| A |   |   | ^ |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+

None


ValueError: Incorrect action specified: None

## Evaluation

In [ ]:
import io
from contextlib import redirect_stdout

SUCCESS_MSG = 'AA has successfully navigated the road!'
FAILURE_MSG = 'There is a crash on the road!'


def _capture_run(fn, verbose=False):
    """Run a callable, capture notebook output, and swallow SystemExit used by env.step()."""
    buffer = io.StringIO()
    try:
        with redirect_stdout(buffer):
            fn()
    except SystemExit:
        pass
    output = buffer.getvalue()
    if verbose:
        print(output)
    return output



def _scaled_score(raw_score, max_cases=7, max_points=10):
    scaled = round((raw_score / max_cases) * max_points, 1)
    return min(scaled, max_points)



def _run_case(agent_type, case, verbose=False):
    driver = AutonomousDriving(
        agent_type=agent_type,
        height=case['height'],
        width=case['width'],
        occupancy=case['occupancy'],
        init=case['init'],
        seed=case['seed'],
    )
    return _capture_run(lambda: driver.run(agent_type), verbose=verbose)

def grade_q2(tests, verbose=False):
    print('----- Testing Q2 -----')
    score_q2 = 0
    num_test_cases = len(tests['q2'])

    for test_case in range(num_test_cases):
        case = tests['q2'][str(test_case)]
        out = _run_case('expectimax', case, verbose=verbose)

        if SUCCESS_MSG in out:
            score_q2 += 1
            print(f'Testcase {test_case}: PASS')
        elif FAILURE_MSG in out:
            print(f'Testcase {test_case}: FAIL')
        else:
            print(f'Testcase {test_case}: FAIL')
            print('There was an error in running test case: ', test_case)

    return _scaled_score(score_q2)


#### Test Cases

In [ ]:

tests = {
"q2": {
    "0":  {
        "height": 5,
        "width": 5,
        "occupancy": 0.1,
        "init": 0.2,
        "seed": 33
        },
    "1": {
        "height": 5,
        "width": 5,
        "occupancy": 0.1,
        "init": 0.2,
        "seed": 16
        },
    "2": {
        "height": 5,
        "width": 4,
        "occupancy": 0.1,
        "init": 0.1,
        "seed": 12
        },
    "3": {
        "height": 5,
        "width": 4,
        "occupancy": 0.1,
        "init": 0.1,
        "seed": 5
        },
    "4": {
        "height": 5,
        "width": 3,
        "occupancy": 0.1,
        "init": 0.1,
        "seed": 12
        },
    "5": {
        "height": 5,
        "width": 3,
        "occupancy": 0.1,
        "init": 0.1,
        "seed": 19
        },
    "6": {
        "height": 5,
        "width": 3,
        "occupancy": 0.1,
        "init": 0.2,
        "seed": 21
        },
    "7": {
        "height": 5,
        "width": 2,
        "occupancy": 0.1,
        "init": 0.2,
        "seed": 33
        },
    "8": {
        "height": 5,
        "width": 2,
        "occupancy": 0.1,
        "init": 0.2,
        "seed": 36
        },
    "9": {
        "height": 2,
        "width": 5,
        "occupancy": 0.3,
        "init": 0.4,
        "seed": 6
        }
    }
}


### Grade Q2

In [ ]:
q2_results = grade_q2(tests, verbose=False)

print('===========================')
print(f'Q2 score:	{q2_results} / 10.0')
print('===========================')


----- Testing Q2 -----


ValueError: Incorrect action specified: None